# Running the MCP GEE Server Standalone

This notebook illustrates how to use the standalone GEE MCP server by calling `GEEMCPClient`.

Make sure you have installed the requirements and set up your environmental variables (`GEMINI_API_KEY`, `GEE_PROJECT` etc).


In [1]:
import os
os.environ['GEE_PROJECT']='ee-raulramos'            # your Google Earth Engine project
os.environ['VERTEXAI_PROJECT']='genai-dev-454121'   # your VertexAI GCP project

import asyncio
from client import GEEMCPClient


In [ ]:
async def run_example():
    async with GEEMCPClient() as client:
        # 1. List datasets
        print("Listing datasets...")
        datasets = await client.list_datasets()
        print(f"Found {len(datasets)} datasets.")
        
        # 2. Get metadata for one dataset
        if datasets:
            dataset_id = datasets[0]['id'] if isinstance(datasets[0], dict) and 'id' in datasets[0] else datasets[0]
            print(f"\nFetching metadata for {dataset_id}...")
            meta = await client.get_metadata(dataset_id)
            print(f"Metadata keys: {list(meta.keys())}")

# Run it
# Modern Jupyter Notebooks run in an event loop and allow top-level awaits.
datasets = await run_example()

# For standard python scripts, use:
# if __name__ == "__main__":
#     asyncio.run(run_example())


Listing datasets...


## Executing GEE Python Code

The server allows executing arbitrary GEE Python code via `execute_gee_python`.

The code **must** define a function named `gee_main()` that returns a tuple/list (e.g., `(result, map)`). The server will run `gee_main()` and return the first element in the JSON response under key `"result"`.


In [5]:
async def run_code_example():
    async with GEEMCPClient() as client:
        code = """
import ee

def gee_main():
    # Simple GEE code
    Point = ee.Geometry.Point([-122.45, 37.75])
    return Point.getInfo(), None
"""
        print("\nExecuting GEE Python code...")
        result = await client.execute_gee_python(code)
        print(f"Execution Result: {result}")

await run_code_example()



Executing GEE Python code...
Execution Result: {'result': {'type': 'Point', 'coordinates': [-122.45, 37.75]}}


## Advanced Example: Sentinel-2 Cloud Percentage and Interactive Map

This example demonstrates how to use `execute_gee_python` to run complex server-side analysis (Sentinel-2 cloud percentage calculation) and return a time-limited **Tile URL** that can be displayed in interactive maps like `ipyleaflet` or `folium` directly in the notebook.

We will analyze a 20km buffer around Barcelona during August 2024.


In [6]:
async def run_advanced_example():
    async with GEEMCPClient() as client:
        # Define server-side script
        code = """
import ee

def gee_main():
    # Barcelona coordinates
    lon, lat = 2.1734, 41.3851
    region = ee.Geometry.Point([lon, lat]).buffer(20000)
    
    # Filter Sentinel-2 SR
    col = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterDate('2024-01-01', '2024-02-28') \
        .filterBounds(region)
    
    # Create composite (median)
    img = col.median()
    
    # SCL band cloud mask (8: Cloud Medium, 9: Cloud High, 10: Thin Cirrus)
    scl = img.select('SCL')
    cloud_mask = scl.eq(8).Or(scl.eq(9)).Or(scl.eq(10))
    
    # Calculate Cloudy Area
    pixel_area = ee.Image.pixelArea().updateMask(cloud_mask)
    cloud_area = pixel_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=20,
        maxPixels=1e9
    ).get('area').getInfo() or 0.0
    
    total_area = region.area().getInfo()
    percentage = (cloud_area / total_area) * 100
    
    # Get Map Tile URL
    vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
    map_id = img.getMapId(vis_params)
    tile_url = map_id['tile_fetcher'].url_format
    
    return {
        'cloud_percentage': percentage,
        'tile_url': tile_url,
        'total_area_km2': total_area / 1e6,
        'cloud_area_km2': cloud_area / 1e6
    }, None
"""
        response = await client.execute_gee_python(code)
        
        # response is already parsed by client.py _parse_result
        res = response
        
        if 'errors' in res:
            print(f"Error: {res['errors']}")
            return None
        else:
            data = res['result']
            print(f"Cloud Percentage around Barcelona in Jan-Feb 2024: {data['cloud_percentage']:.2f}%")
            print(f"Total Area: {data['total_area_km2']:.2f} km²")
            print(f"Cloud Area: {data['cloud_area_km2']:.2f} km²")
            print(f"\\nTile URL for Map Visualization retrieved.")
            return data['tile_url']

tile_url = await run_advanced_example()


Cloud Percentage around Barcelona in Jan-Feb 2024: 13.18%
Total Area: 1241.62 km²
Cloud Area: 163.70 km²
\nTile URL for Map Visualization retrieved.


In [7]:
import folium

if tile_url:
    # Create a map centered on Barcelona
    m = folium.Map(location=[41.3851, 2.1734], zoom_start=11)

    # Add the Tile URL from GEE
    folium.TileLayer(
        tiles=tile_url,
        attr='Google Earth Engine',
        name='Sentinel-2',
        overlay=True,
        control=True
    ).add_to(m)

    folium.LayerControl().add_to(m)

    # Display the map
    display(m)
else:
    print("Could not display map (tile_url is None)")
